In [ ]:
import os

ROOT_DIR = "/kaggle/working"
MODELS_DIR = os.path.join(ROOT_DIR, "models")

os.makedirs(MODELS_DIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# CELL: DOWNLOAD PIPER MODELS (NO AUTH, STRICT FILTER)
# ---------------------------------------------------------------------------
import os
from huggingface_hub import snapshot_download

# --- DIRECTORY SETUP ---
ROOT_DIR = "/kaggle/working"
MODELS_DIR = os.path.join(ROOT_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

print("⬇️ Downloading Piper English Models (ONNX/JSON only)...")

try:
    snapshot_download(
        repo_id="rhasspy/piper-voices",
        repo_type="model",
        # STRICT FILTERING: Only download .onnx and .json files inside the 'en' folder.
        # This prevents the script from listing or touching the thousands of .wav sample files
        # which is what triggers the anonymous rate limit.
        allow_patterns=["en/**/*.onnx", "en/**/*.json"],
        local_dir=MODELS_DIR,
        local_dir_use_symlinks=False
    )
    print(f"✅ Models downloaded to {MODELS_DIR}")
except Exception as e:
    print(f"❌ Model download failed: {e}")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 3: CONFIGURATION & SOURCE SETUP
# ---------------------------------------------------------------------------
import os
import sys
import shutil

# Hardcoded Dataset Path
PYAPI_DATASET_NAME = "sage-pyapi-code"
PYAPI_SOURCE_DIR = f"/kaggle/input/{PYAPI_DATASET_NAME}"
WORK_DIR = "/kaggle/working/pyapi" # The code folder, sibling to models

if not os.path.exists(os.path.join(PYAPI_SOURCE_DIR, "main.py")):
    print(f"❌ Error: 'main.py' not found in {PYAPI_SOURCE_DIR}")
    sys.exit(1)

# Copy Code
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

print(f"📂 Copying PyAPI code to {WORK_DIR}...")
shutil.copytree(PYAPI_SOURCE_DIR, WORK_DIR)

print(f"✅ PyAPI Source Configured.")

# Zrok Token
ZROK_TOKEN_PATH = "/kaggle/input/sage-zrok-token/.zrok_api_key"
zrok_token = None

if os.path.isfile(ZROK_TOKEN_PATH):
    with open(ZROK_TOKEN_PATH, "r") as f:
        zrok_token = f.read().strip()

if not zrok_token:
    from kaggle_secrets import UserSecretsClient
    try:
        zrok_token = UserSecretsClient().get_secret("zrok_token")
    except:
        pass

if not zrok_token:
    print("❌ Zrok Token not found!")
    sys.exit(1)

In [ ]:
# ---------------------------------------------------------------------------
# CELL 5: VENV & INSTALLATION
# ---------------------------------------------------------------------------
VENV_PATH = os.path.join(WORK_DIR, "venv")
!python3 -m venv "{VENV_PATH}" --without-pip

VENV_BIN = os.path.join(VENV_PATH, "bin")
VENV_PYTHON = os.path.join(VENV_BIN, "python")
VENV_PIP = os.path.join(VENV_BIN, "pip")

print("🔧 Bootstrapping pip...")
!wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py
!{VENV_PYTHON} /tmp/get-pip.py > /dev/null

print("📦 Installing Dependencies...")
req_path = os.path.join(WORK_DIR, "requirements_server_gpu.txt")
!{VENV_PIP} install wrapt --no-cache-dir
!{VENV_PIP} install -r "{req_path}" --no-cache-dir
!{VENV_PIP} install pyngrok nest_asyncio > /dev/null

In [ ]:
# ---------------------------------------------------------------------------
# CELL 6: ZROK SETUP
# ---------------------------------------------------------------------------
if not os.path.exists("zrok"):
    !wget https://github.com/openziti/zrok/releases/download/v1.1.3/zrok_1.1.3_linux_amd64.tar.gz > /dev/null
    !tar -xzf zrok_1.1.3_linux_amd64.tar.gz > /dev/null
    !chmod +x zrok

!./zrok disable > /dev/null 2>&1
!./zrok enable --headless "$zrok_token"

In [ ]:
# ---------------------------------------------------------------------------
# CELL 7: START SERVER (VENV ONLY)
# ---------------------------------------------------------------------------
import threading
import subprocess
import time
import os

SERVER_PORT = 8009
WORK_DIR = "/kaggle/working/pyapi"

# Virtual Environment Paths
VENV_PATH = os.path.join(WORK_DIR, "venv")
VENV_BIN = os.path.join(VENV_PATH, "bin")
VENV_PYTHON = os.path.join(VENV_BIN, "python")

def run_pyapi_server():
    """Runs the FastAPI server using the VENV python"""
    
    # Command to run uvicorn
    cmd = [
        VENV_PYTHON, "-m", "uvicorn", "main:app", 
        "--host", "0.0.0.0", 
        "--port", str(SERVER_PORT)
    ]
    
    # Setup Environment
    env = os.environ.copy()
    env["VIRTUAL_ENV"] = VENV_PATH
    
    # Add venv/bin to PATH so 'piper' (installed via pip) can be found by shutil.which()
    env["PATH"] = f"{VENV_BIN}:" + env["PATH"]
    
    # Strip PYTHONPATH to ensure isolation from Kaggle system libs
    if "PYTHONPATH" in env: 
        del env["PYTHONPATH"]

    print(f"🚀 Starting Server in {WORK_DIR}...")
    
    # Run process
    # cwd=WORK_DIR ensures the app runs from /kaggle/working/pyapi
    # This is crucial for the service to find '../models' (which is /kaggle/working/models)
    subprocess.Popen(
        cmd, 
        cwd=WORK_DIR, 
        env=env
    )

def start_zrok_tunnel():
    """Starts the Zrok public share"""
    print("⏳ Waiting for server to warm up...")
    time.sleep(10)
    
    print("🚇 Opening Zrok Tunnel...")
    # Start zrok share
    subprocess.Popen(
        ["./zrok", "share", "public", f"localhost:{SERVER_PORT}", "--headless"],
        stdout=subprocess.DEVNULL, # Keep notebook clean
        stderr=subprocess.DEVNULL
    )
    
    time.sleep(10)
    
    # Display the public URL
    print("\n" + "="*40)
    print(" 🌐 ZROK PUBLIC ACCESS ")
    print("="*40)
    subprocess.run(["./zrok", "overview"])
    print("="*40)

# 1. Start Server Thread
threading.Thread(target=run_pyapi_server, daemon=True).start()

# 2. Start Tunnel Thread
threading.Thread(target=start_zrok_tunnel, daemon=True).start()

# 3. Keep-Alive Loop
print("✨ System operational. Running keep-alive loop.")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("🛑 Shutting down.")